# Steam Games – Data Mining Project
## Section 2: Data Preprocessing & Feature Engineering

**Team 9 – Brewed Clusters**

---

### What does this notebook do?

This notebook takes the raw CSV from Section 1 and produces a **clean, model-ready dataset**.

Steps covered:
1. Load the data with a corrected CSV routine (fixes a column-misalignment bug)
2. Drop columns with very high missing rates
3. Filter games with fewer than 10 reviews
4. Parse `Estimated owners` string ranges to numeric midpoints
5. Parse multi-value fields (Genres, Tags, Categories) into binary columns
6. Derive `review_ratio` and create the classification label
7. Engineer new features (log-price, release era, language count)
8. Impute remaining missing values
9. Standardise continuous features
10. Export the cleaned dataset to Parquet

The Parquet file produced here is the **single source of truth** for all modelling notebooks.

---
## Step 0 – Import Libraries

In [ ]:
import csv
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 60)
sns.set_theme(style='whitegrid')

print('Libraries loaded successfully!')

---
## Step 1 – Load Data with Corrected CSV Routine

**Why not `pd.read_csv()`?**  
The `Supported languages` column stores values like `['English', 'French']` — a comma-separated list inside quotes.  
`pd.read_csv()` misreads this and silently shifts every column from index 9 onward by one position.  
This means `Peak CCU`, `Price`, and most other columns contain wrong values when loaded with the default reader.

**The fix:** use Python's built-in `csv.reader`, which correctly handles quoted fields,  
then detect any over-split rows (40 fields instead of 39) and merge the split field back.

In [ ]:
DATA_PATH = 'Data/games.csv'

rows_data = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        # If a row has 40 fields, the Supported languages column was split
        # Merge fields 9 and 10 back into a single field
        if len(row) == 40:
            row = row[:9] + [row[9] + ',' + row[10]] + row[11:]
        rows_data.append(row)

df = pd.DataFrame(rows_data, columns=header)

print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Sanity check: verify the fix worked by inspecting Peak CCU
# After the fix, Peak CCU should have large values for popular games
df['Peak CCU'] = pd.to_numeric(df['Peak CCU'], errors='coerce')
df['Positive'] = pd.to_numeric(df['Positive'], errors='coerce')
df['Negative'] = pd.to_numeric(df['Negative'], errors='coerce')
df['Price']    = pd.to_numeric(df['Price'], errors='coerce')

print(f"Peak CCU max  : {df['Peak CCU'].max():,.0f}  (should be in the millions for CS2/Dota2)")
print(f"Price max     : {df['Price'].max():,.2f}")
print(f"Positive max  : {df['Positive'].max():,.0f}")

---
## Step 2 – Drop High-Missingness Columns

Columns with very high missing rates carry no useful signal for modelling.  
We drop any column missing more than **50%** of its values, plus non-predictive metadata columns.

In [ ]:
# Columns to drop: either >50% missing, or not useful for prediction
cols_to_drop = [
    'Movies',           # 100% missing
    'Score rank',       # 99.97% missing
    'Metacritic url',   # 96.5% missing
    'Reviews',          # 90% missing (text snippets, not structured)
    'Notes',            # 81.7% missing
    'Website',          # 59.5% missing
    'Support url',      # 55.8% missing
    'Support email',    # not predictive
    'Header image',     # URL, not predictive
    'Screenshots',      # URL, not predictive
    'Metacritic url',   # already listed, safe to repeat
    'AppID',            # identifier only
]

# Only drop columns that actually exist in the dataframe
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)

print(f'Columns remaining: {df.shape[1]}')
print(list(df.columns))

---
## Step 3 – Filter Games with Sufficient Reviews

We need to exclude games that cannot be reliably labelled.

- Games with **0 reviews**: no label possible.
- Games with **fewer than 10 reviews**: Steam itself does not assign a reception label 
  (e.g. "Mostly Positive") until a game reaches 10 reviews. A ratio from 1–9 reviews 
  is statistically unreliable.

We apply a minimum threshold of **10 total reviews**.

In [ ]:
total_before = len(df)

# Total reviews = Positive + Negative
df['total_reviews'] = df['Positive'] + df['Negative']

df = df[df['total_reviews'] >= 10].reset_index(drop=True)

total_after = len(df)
print(f'Before filter : {total_before:,} games')
print(f'After filter  : {total_after:,} games')
print(f'Removed       : {total_before - total_after:,} games with fewer than 10 reviews')

---
## Step 4 – Parse `Estimated owners` to Numeric Midpoints

The `Estimated owners` column stores values as string ranges like `"50000 - 100000"`.  
We convert each range to its **numeric midpoint** (e.g. 75,000) for use as a feature.

In [ ]:
def parse_owners_midpoint(s):
    """Convert '50000 - 100000' -> 75000.0"""
    try:
        parts = str(s).replace(',', '').split('-')
        low, high = float(parts[0].strip()), float(parts[1].strip())
        return (low + high) / 2
    except:
        return np.nan

df['owners_midpoint'] = df['Estimated owners'].apply(parse_owners_midpoint)

print('Owners midpoint — sample values:')
print(df[['Estimated owners', 'owners_midpoint']].head(10))

---
## Step 5 – Derive `review_ratio` and Create the Classification Label

- `review_ratio` = Positive / (Positive + Negative)
- **Good** = ratio ≥ 0.70 (≥ 70% positive reviews)
- **Bad** = ratio < 0.70

The 0.70 threshold aligns with Steam's own definition of "Mostly Positive" reception.

In [ ]:
df['review_ratio'] = df['Positive'] / df['total_reviews']

df['label'] = (df['review_ratio'] >= 0.70).astype(int)
# 1 = Good, 0 = Bad

label_counts = df['label'].value_counts()
label_pct    = df['label'].value_counts(normalize=True) * 100

print('Label distribution:')
print(f'  Good (1) : {label_counts[1]:>7,}  ({label_pct[1]:.1f}%)')
print(f'  Bad  (0) : {label_counts[0]:>7,}  ({label_pct[0]:.1f}%)')

# Bar chart
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Good (>=70%)', 'Bad (<70%)'], [label_counts[1], label_counts[0]],
       color=['steelblue', 'salmon'])
ax.set_title('Class distribution after filtering')
ax.set_ylabel('Number of games')
for i, v in enumerate([label_counts[1], label_counts[0]]):
    ax.text(i, v + 100, f'{v:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## Step 6 – Parse Multi-Value Fields (Genres, Tags, Categories)

These columns store multiple values as comma-separated strings:  
`"Action,Indie,RPG"` → we split and binarise each value into its own 0/1 column.

- **Genres** and **Categories**: keep all values
- **Tags**: keep only the **top 50 by frequency** to avoid a very wide, sparse feature matrix

In [ ]:
def parse_list_column(series):
    """Split a comma-separated string column into a list of stripped values."""
    return series.fillna('').apply(
        lambda x: [v.strip() for v in x.split(',') if v.strip()]
    )

# ── Genres ────────────────────────────────────────────────────────────────────
genres_lists = parse_list_column(df['Genres'])
mlb_genres   = MultiLabelBinarizer()
genres_dummies = pd.DataFrame(
    mlb_genres.fit_transform(genres_lists),
    columns=[f'genre_{g}' for g in mlb_genres.classes_]
)

# ── Categories ────────────────────────────────────────────────────────────────
cats_lists   = parse_list_column(df['Categories'])
mlb_cats     = MultiLabelBinarizer()
cats_dummies = pd.DataFrame(
    mlb_cats.fit_transform(cats_lists),
    columns=[f'cat_{c}' for c in mlb_cats.classes_]
)

# ── Tags (top 50 only) ────────────────────────────────────────────────────────
tags_lists = parse_list_column(df['Tags'])

# Find the 50 most frequent tags across all games
from collections import Counter
tag_counts = Counter(tag for tags in tags_lists for tag in tags)
top50_tags = {tag for tag, _ in tag_counts.most_common(50)}

# Keep only top-50 tags per game
tags_filtered = tags_lists.apply(lambda tags: [t for t in tags if t in top50_tags])
mlb_tags      = MultiLabelBinarizer(classes=sorted(top50_tags))
tags_dummies  = pd.DataFrame(
    mlb_tags.fit_transform(tags_filtered),
    columns=[f'tag_{t}' for t in mlb_tags.classes_]
)

print(f'Genre columns    : {genres_dummies.shape[1]}')
print(f'Category columns : {cats_dummies.shape[1]}')
print(f'Tag columns      : {tags_dummies.shape[1]}')

---
## Step 7 – Feature Engineering

We derive new features and transform existing ones:

| New feature | Source | Transformation |
|-------------|--------|----------------|
| `log_price` | Price | log₁₀(Price + 1) — reduces right skew |
| `release_era` | Release date | Year → era bucket |
| `n_languages` | Supported languages | Count of languages listed |

In [ ]:
# ── Log-price ─────────────────────────────────────────────────────────────────
df['Price'] = pd.to_numeric(df['Price'], errors='coerce').fillna(0)
df['log_price'] = np.log1p(df['Price'])   # log(1 + price)

# ── Release era ───────────────────────────────────────────────────────────────
df['release_year'] = pd.to_datetime(df['Release date'], errors='coerce').dt.year

def year_to_era(y):
    if pd.isna(y):    return 'Unknown'
    if y < 2015:      return 'pre-2015'
    if y <= 2019:     return '2015-2019'
    return '2020+'

df['release_era'] = df['release_year'].apply(year_to_era)
df = pd.get_dummies(df, columns=['release_era'], prefix='era')

# ── Number of supported languages ─────────────────────────────────────────────
def count_languages(s):
    """Count the number of languages in the supported languages string."""
    if pd.isna(s) or str(s).strip() in ('', '[]'):
        return 0
    return len([v for v in str(s).split(',') if v.strip()])

df['n_languages'] = df['Supported languages'].apply(count_languages)

# ── Platform flags ────────────────────────────────────────────────────────────
for col in ['Windows', 'Mac', 'Linux']:
    df[col] = df[col].map({'True': 1, 'False': 0, True: 1, False: 0}).fillna(0).astype(int)

# ── Other numeric columns ─────────────────────────────────────────────────────
numeric_cols = [
    'Required age', 'DiscountDLC count', 'Achievements',
    'Average playtime forever', 'Median playtime forever',
    'Recommendations', 'Metacritic score'
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print('Feature engineering complete.')
print(f"Era columns : {[c for c in df.columns if c.startswith('era_')]}")
print(f"log_price sample : {df['log_price'].describe().round(2).to_dict()}")

---
## Step 8 – Assemble Final Feature Matrix

We now combine all feature groups into one clean DataFrame,  
keeping only the columns needed for modelling.

In [ ]:
# ── Numeric features to keep ──────────────────────────────────────────────────
base_numeric = [
    'log_price',
    'Required age',
    'DiscountDLC count',
    'Achievements',
    'Average playtime forever',   # NOTE: potential post-release leakage — may be removed later
    'Median playtime forever',    # NOTE: potential post-release leakage — may be removed later
    'Recommendations',            # NOTE: potential post-release leakage — may be removed later
    'Metacritic score',
    'Windows', 'Mac', 'Linux',
    'n_languages',
]

era_cols  = [c for c in df.columns if c.startswith('era_')]

# Reset indices before concatenating so rows align correctly
df_reset       = df.reset_index(drop=True)
genres_dummies = genres_dummies.reset_index(drop=True)
cats_dummies   = cats_dummies.reset_index(drop=True)
tags_dummies   = tags_dummies.reset_index(drop=True)

feature_cols = base_numeric + era_cols
X = pd.concat(
    [df_reset[feature_cols], genres_dummies, cats_dummies, tags_dummies],
    axis=1
)
y = df_reset['label']

print(f'Feature matrix shape : {X.shape}')
print(f'Label shape          : {y.shape}')
print(f'Class balance        : {y.value_counts(normalize=True).round(3).to_dict()}')

---
## Step 9 – Impute Remaining Missing Values

Any remaining NaN values in numeric columns are filled with the **column median**.  
Median is more robust than mean for skewed distributions.

In [ ]:
missing_before = X.isnull().sum().sum()

# Fill numeric columns with their median
for col in X.select_dtypes(include='number').columns:
    X[col] = X[col].fillna(X[col].median())

missing_after = X.isnull().sum().sum()

print(f'Missing values before imputation : {missing_before:,}')
print(f'Missing values after  imputation : {missing_after:,}')

---
## Step 10 – Standardise Continuous Features

`StandardScaler` transforms each continuous feature to have mean = 0 and std = 1.  
This is important for Logistic Regression and SVM but does not affect tree-based models.

We only scale continuous columns — binary dummy columns (0/1) are left as-is.

In [ ]:
# Identify continuous columns to scale (not binary 0/1 dummies)
continuous_cols = [
    'log_price', 'Required age', 'DiscountDLC count', 'Achievements',
    'Average playtime forever', 'Median playtime forever',
    'Recommendations', 'Metacritic score', 'n_languages'
]
continuous_cols = [c for c in continuous_cols if c in X.columns]

scaler = StandardScaler()
X[continuous_cols] = scaler.fit_transform(X[continuous_cols])

print('Scaling complete.')
print(X[continuous_cols].describe().T[['mean', 'std']].round(3))

---
## Step 11 – Export to Parquet

We save both the feature matrix `X` and labels `y` to Parquet files.  
All modelling notebooks (Sections 4a, 4b, 4c) will load from these files — **do not re-run preprocessing from scratch for each model**.

In [ ]:
import os
os.makedirs('Data/processed', exist_ok=True)

X.to_parquet('Data/processed/X_classification.parquet', index=False)
y.to_frame().to_parquet('Data/processed/y_classification.parquet', index=False)

print('Saved:')
print(f'  Data/processed/X_classification.parquet  — {X.shape}')
print(f'  Data/processed/y_classification.parquet  — {y.shape}')

---
## Section 2 Summary

| Step | Action | Result |
|------|--------|--------|
| CSV loading | Corrected column-misalignment bug | All columns correctly aligned |
| Column dropping | Removed >50% missing + non-predictive columns | Cleaner dataset |
| Review filter | Kept games with ≥ 10 reviews | Reliable labels only |
| Label derivation | `review_ratio ≥ 0.70` → Good / Bad | Binary classification target |
| Multi-value parsing | Genres, Categories, Tags → binary columns | Top-50 tags retained |
| Feature engineering | log_price, release_era, n_languages | Reduced skew, structured features |
| Imputation | Median fill for remaining NaN values | No missing values in feature matrix |
| Scaling | StandardScaler on continuous columns | Mean=0, Std=1 |
| Export | Parquet files to `Data/processed/` | Ready for modelling notebooks |

**Next step:** `section3_eda.ipynb` — Exploratory Data Analysis